In [1]:
import pandas as pd
import os

In [2]:
df = pd.read_excel('./nathan_to_fix.xlsx')

In [3]:
images = df.target_filename.dropna().unique()

In [4]:
import dropbox
import os
import numpy as np

In [45]:
# token = os.getenv('dropbox_access_token')
# dbx = dropbox.Dropbox(token)

# base_path = 'Nanonets/analysis'
# os.makedirs(f"{base_path}", exist_ok=True)

# images = [
#     x for x in file["target_filename"].unique()
#     if x is not None and x == x
# ]

# created_folders = set()

# def get_folder(image_name):
#     return image_name.split('_')[2]

# for image in images:
#     folder = get_folder(image)
#     if folder not in created_folders:
#         os.makedirs(f"{base_path}/{folder}", exist_ok=True)
#         created_folders.add(folder)
#     try:
#         dropbox_path = f"/Ireland Maps/{base_path}/{folder}/{image}"
#         local_path = f"{base_path}/{folder}/{image}"
#         dbx.files_download_to_file(local_path, dropbox_path)
#     except Exception as e:
#         print(f"Error downloading {image}: {e}")


In [49]:
'''
task: identify 1-page townland images
one_page_townlands = a dictionary with the townland key as the key and the set of image names as the value
res = a deque of all images names
for every row in df:
    - create a townland key: (county, barony, parish, townland, county_gv, barony_gv, parish_gv, townland_gv)
    - if the townland key is not in the dictionary, add it and set the value to the df[target_filename]
    - if the townland key is in the dictionary, add the df[target_filename] to the set of values for the townland key

for every key in one_page_townlands:
    - if the length of the set of values for the key is > 1, remove the df[target_filename] from the deque

- save res to a json file
'''

from collections import defaultdict
import json

# Build townland -> images mapping
townland_to_images = defaultdict(set)

for _, row in df.iterrows():
    townland_key = (

        row["county"],
        row["barony"],
        row["parish"],
        row["townland"],
    )

    townland_to_images[townland_key].add(row["target_filename"])

rows = []
for key, images in townland_to_images.items():
    # Convert tuple key to a string representation
    key_str = str(key)  # or str(key) if you prefer
    for image in images:
        rows.append([key_str, image])

# Create a DataFrame
output_df = pd.DataFrame(rows, columns=["Townland Key", "Page"])
output_df.sort_values(by="Page", inplace=True)
# Write to Excel
output_df.to_excel("townland_images.xlsx", index=False)


# Identify all images involved in multi-page townlands
bad_images = set()

for images in townland_to_images.values():
    if len(images) > 1:
        bad_images.update(images)

# All images in the dataset
all_images = set(df["target_filename"].dropna().unique())

print(len(all_images))

#  Keep only clean 1-page townland images
res = sorted(all_images - bad_images)
print(len(res))

# Save result to JSON
with open("one_page_townland_images.json", "w") as f:
    json.dump(res, f, indent=2)

1904
653


In [6]:
df.head()

,filename,county,barony,parish,townland,county_gv,barony_gv,parish_gv,townland_gv,adm,check_townland,check_page,sum_land_val_num,total_land_val_num,sum_total_val_num,total_total_val_num,folder,page,target_filename
0,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNAFF,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNAFF,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN CARNAFF",0,1,196.80,177.50,224.00,224.00,4.0,65.0,IRE_GRIFF_004_065.jpg
1,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNCOGGY,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNCOGGY,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN CARNCOGGY",0,1,144.50,146.50,173.75,173.75,4.0,65.0,IRE_GRIFF_004_065.jpg
2,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DEEPSTOWN,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DEEPSTOWN,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN DEEPSTOWN",1,1,82.50,82.50,86.75,6.75,4.0,65.0,IRE_GRIFF_004_065.jpg
3,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DERRYKEIGHAN,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DERRYKEIGHAN,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN DERRYKEIGHAN",0,1,166.15,135.90,185.05,170.55,4.0,65.0,IRE_GRIFF_004_065.jpg
4,IRE_GRIFF_004_067.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,BALLYDIVITY,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,BALLYDIVITY,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN BALLYDIVITY",1,1,86.15,210.65,279.50,282.50,4.0,67.0,IRE_GRIFF_004_067.jpg


In [17]:
for image in images:
    townlands_to_check = df[(df['target_filename'] == image) & (df['check_townland'] == 1)].townland.unique()
    print(townlands_to_check)
    # res = subset.filter(subset['check_townland'] == 1).townland.unique()
    # print(res)
    break

['DEEPSTOWN']
